## Comparaison de différents modèles supervisés

### A réaliser :
* Pour chaque algorithme que vous allez tester, vous devez :
    * Réaliser au préalable une séparation en jeu d'apprentissage et jeu de test via une validation croisée.
    * Si les features quantitatives que vous souhaitez utiliser ont des ordres de grandeur très différents les uns des autres, et que vous utilisez un algorithme de regression qui est sensible à cette différence, alors il faut réaliser un scaling (normalisation) de la donnée au préalable.
    * Entrainer le modèle sur le jeu de Train
    * Prédire la cible sur la donnée de test (nous appelons cette étape, l'inférence).
    * Calculer les métriques de performance R2, MAE et RMSE sur le jeu de train et de test.
    * Interpréter les résultats pour juger de la fiabilité de l'algorithme.

* Déterminer le modèle le plus performant parmi ceux testés.

In [96]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
#Selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV, 
    cross_validate,
)
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error 
from sklearn.inspection import permutation_importance

#Preprocess
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor


In [97]:
X_OHE = pd.read_csv(r"C:\Users\ethan\OpenClassroom\Projet_2\Projet\Anticipez_les_besoins_\Fichiers\data_etape_3_clean_X.csv", keep_default_na=False)
y = pd.read_csv(r"C:\Users\ethan\OpenClassroom\Projet_2\Projet\Anticipez_les_besoins_\Fichiers\data_etape_3_clean_y.csv", keep_default_na=False)

### --- Modèle 1 : DummyRegressor ---


Copies dédiées

In [98]:
X_dummy = X_OHE.copy()
y_dummy = y.copy()

Séparation apprentissage test

In [99]:
Xtr, Xte, ytr, yte = train_test_split(X_dummy, y_dummy, test_size=0.2, random_state=42)

Validation croisée sur le jeu d’apprentissage

In [100]:
scoring = {"r2":"r2", "mae":"neg_mean_absolute_error", "rmse":"neg_root_mean_squared_error"}
cv_res = cross_validate(DummyRegressor(strategy="median"), Xtr, ytr, cv=5, scoring=scoring, return_train_score=False)

cv_r2   = cv_res["test_r2"].mean()
cv_mae  = -cv_res["test_mae"].mean()
cv_rmse = -cv_res["test_rmse"].mean()

Entraînement sur tout le train

In [101]:
dum = DummyRegressor(strategy="median")
dum.fit(Xtr, ytr)

DummyRegressor(strategy='median')

Inférence

In [102]:
yp_tr = dum.predict(Xtr)
yp_te = dum.predict(Xte)

Métriques apprentissage

In [103]:
r2_tr   = r2_score(ytr, yp_tr)
mae_tr  = mean_absolute_error(ytr, yp_tr)
rmse_tr = root_mean_squared_error(ytr, yp_tr)

Métriques test

In [104]:
r2_te   = r2_score(yte, yp_te)
mae_te  = mean_absolute_error(yte, yp_te)
rmse_te = root_mean_squared_error(yte, yp_te)

In [105]:

print(f"Dummy, CV R2 moyen {cv_r2:.3f}, CV MAE moyen {cv_mae:.2f}, CV RMSE moyen {cv_rmse:.2f}")
print(f"Dummy, Train  R2 {r2_tr:.3f}, MAE {mae_tr:.2f}, RMSE {rmse_tr:.2f}")
print(f"Dummy, Test   R2 {r2_te:.3f}, MAE {mae_te:.2f}, RMSE {rmse_te:.2f}")


Dummy, CV R2 moyen -0.124, CV MAE moyen 1595297.94, CV RMSE moyen 2330765.60
Dummy, Train  R2 -0.123, MAE 1594893.67, RMSE 2334513.25
Dummy, Test   R2 -0.156, MAE 1632270.78, RMSE 2296427.92


### --- Modèle 2 : Régression linéaire avec scaling ---



Copies dédiées


In [106]:
X_lin = X_OHE.copy()
y_lin = y.copy()



Séparation apprentissage et test


In [107]:
Xtr, Xte, ytr, yte = train_test_split(X_lin, y_lin, test_size=0.2, random_state=42)



Estimateur enveloppe, scaling puis modèle


In [108]:
class ScaleRegressor:
    def __init__(self, model):
        self.model = model
        self.scaler = StandardScaler()
        self.cols_ = None
    def fit(self, X_fit, y_fit):
        self.cols_ = X_fit.columns
        Xs = self.scaler.fit_transform(X_fit)
        self.model.fit(Xs, y_fit)
        return self
    def predict(self, X_new):
        Xs = self.scaler.transform(X_new[self.cols_])
        return self.model.predict(Xs)
    def get_params(self, deep=True):
        return {"model": self.model}
    def set_params(self, **params):
        for k,v in params.items(): setattr(self, k, v)
        return self

est = ScaleRegressor(LinearRegression())



Validation croisée sur le jeu d’apprentissage


In [109]:
scoring = {"r2":"r2", "mae":"neg_mean_absolute_error", "rmse":"neg_root_mean_squared_error"}
cv_res = cross_validate(est, Xtr, ytr, cv=5, scoring=scoring, return_train_score=False)

cv_r2   = cv_res["test_r2"].mean()
cv_mae  = -cv_res["test_mae"].mean()
cv_rmse = -cv_res["test_rmse"].mean()



Entraînement complet


In [110]:
est.fit(Xtr, ytr)


Inférence


In [111]:
yp_tr = est.predict(Xtr)
yp_te = est.predict(Xte)


Métriques


In [112]:
r2_tr   = r2_score(ytr, yp_tr);  mae_tr  = mean_absolute_error(ytr, yp_tr);  rmse_tr = root_mean_squared_error(ytr, yp_tr)
r2_te   = r2_score(yte, yp_te);  mae_te  = mean_absolute_error(yte, yp_te);  rmse_te = root_mean_squared_error(yte, yp_te)

In [113]:

print(f"LIN, CV R2 moyen {cv_r2:.3f}, CV MAE moyen {cv_mae:.2f}, CV RMSE moyen {cv_rmse:.2f}")
print(f"LIN, Train  R2 {r2_tr:.3f}, MAE {mae_tr:.2f}, RMSE {rmse_tr:.2f}")
print(f"LIN, Test   R2 {r2_te:.3f}, MAE {mae_te:.2f}, RMSE {rmse_te:.2f}")


LIN, CV R2 moyen -472186236479247350524542976.000, CV MAE moyen 3153641720498913280.00, CV RMSE moyen 29217420149257289728.00
LIN, Train  R2 0.904, MAE 463674.07, RMSE 683707.25
LIN, Test   R2 -1707813775799989158543360.000, MAE 223582748844991104.00, RMSE 2790826525628413952.00


### --- Modèle 3 : SVR avec scaling ---


Copies dédiées

In [114]:
X_svr = X_OHE.copy()
y_svr = y.copy()

Séparation apprentissage et test

In [115]:
Xtr, Xte, ytr, yte = train_test_split(X_svr, y_svr, test_size=0.2, random_state=42)

Estimateur: SVR dans ton wrapper de scaling

In [116]:
est = ScaleRegressor(SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale"))

Validation croisée sur le jeu d’apprentissage

In [117]:
scoring = {"r2":"r2", "mae":"neg_mean_absolute_error", "rmse":"neg_root_mean_squared_error"}
cv_res = cross_validate(est, Xtr, ytr, cv=5, scoring=scoring, return_train_score=False)

cv_r2   = cv_res["test_r2"].mean()
cv_mae  = -cv_res["test_mae"].mean()
cv_rmse = -cv_res["test_rmse"].mean()

c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), 

Entraînement complet

In [118]:
est.fit(Xtr, ytr)

c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\utils\validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Inférence

In [119]:
yp_tr = est.predict(Xtr)
yp_te = est.predict(Xte)

Métriques

In [120]:
r2_tr   = r2_score(ytr, yp_tr);  mae_tr  = mean_absolute_error(ytr, yp_tr);  rmse_tr = root_mean_squared_error(ytr, yp_tr)
r2_te   = r2_score(yte, yp_te);  mae_te  = mean_absolute_error(yte, yp_te);  rmse_te = root_mean_squared_error(yte, yp_te)

In [121]:
print(f"SVR, CV R2 moyen {cv_r2:.3f}, CV MAE moyen {cv_mae:.2f}, CV RMSE moyen {cv_rmse:.2f}")
print(f"SVR, Train  R2 {r2_tr:.3f}, MAE {mae_tr:.2f}, RMSE {rmse_tr:.2f}")
print(f"SVR, Test   R2 {r2_te:.3f}, MAE {mae_te:.2f}, RMSE {rmse_te:.2f}")

SVR, CV R2 moyen -0.124, CV MAE moyen 1595253.86, CV RMSE moyen 2330707.19
SVR, Train  R2 -0.123, MAE 1594829.29, RMSE 2334413.45
SVR, Test   R2 -0.156, MAE 1632211.44, RMSE 2296344.63


### --- Modèle 4 : RandomForestRegressor ---


Copies dédiées

In [122]:
X_rf = X_OHE.copy()
y_rf = y.copy()


Séparation apprentissage et test

In [123]:
Xtr, Xte, ytr, yte = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42)



Estimateur Forêt Aléatoire


In [124]:
est = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

Validation croisée sur le jeu d’apprentissage

In [125]:
scoring = {"r2":"r2", "mae":"neg_mean_absolute_error", "rmse":"neg_root_mean_squared_error"}
cv_res = cross_validate(est, Xtr, ytr, cv=5, scoring=scoring, return_train_score=False)

cv_r2   = cv_res["test_r2"].mean()
cv_mae  = -cv_res["test_mae"].mean()
cv_rmse = -cv_res["test_rmse"].mean()


c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for exa


Entraînement complet


In [126]:
est.fit(Xtr, ytr)


c:\Users\ethan\Anconda4\Lib\site-packages\sklearn\base.py:1474: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestRegressor(n_estimators=400, n_jobs=-1, random_state=42)


Inférence


In [127]:
yp_tr = est.predict(Xtr)
yp_te = est.predict(Xte)


Métriques


In [128]:
r2_tr   = r2_score(ytr, yp_tr);  mae_tr  = mean_absolute_error(ytr, yp_tr);  rmse_tr = root_mean_squared_error(ytr, yp_tr)
r2_te   = r2_score(yte, yp_te);  mae_te  = mean_absolute_error(yte, yp_te);  rmse_te = root_mean_squared_error(yte, yp_te)

In [129]:
print(f"RF, CV R2 moyen {cv_r2:.3f}, CV MAE moyen {cv_mae:.2f}, CV RMSE moyen {cv_rmse:.2f}")
print(f"RF, Train  R2 {r2_tr:.3f}, MAE {mae_tr:.2f}, RMSE {rmse_tr:.2f}")
print(f"RF, Test   R2 {r2_te:.3f}, MAE {mae_te:.2f}, RMSE {rmse_te:.2f}")

RF, CV R2 moyen 0.946, CV MAE moyen 259969.84, CV RMSE moyen 502711.34
RF, Train  R2 0.993, MAE 93843.02, RMSE 186102.24
RF, Test   R2 0.947, MAE 266682.76, RMSE 490727.09
